# Laboratory 3 — Verification and validation

**Competency:** C4 (Analyse)

Verification asks whether you solved the equations right. Validation asks whether you solved 
the right equations. This laboratory does both: a mesh-convergence study against itself, and 
a comparison against the closed-form solution.

This is the laboratory that matters most. A simulation you have not verified is a picture, 
not a result.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import numpy as np, matplotlib.pyplot as plt
from memslab.plate import Diaphragm, REFERENCE, REFERENCE_PRESSURE
from memslab.morley import convergence_study
import pandas as pd

d, q = REFERENCE, REFERENCE_PRESSURE
table = pd.DataFrame(convergence_study(d, q, refinements=(3, 4, 5, 6)))
table[['eps_w', 'eps_sigma']] *= 100
table = table.rename(columns={'eps_w': 'eps_w [%]', 'eps_sigma': 'eps_sigma [%]'})
print(table.round(3).to_string(index=False))


## Reading the convergence table

Both quantities converge as the mesh is refined, and the relative error, Equation (9), 
falls monotonically. Plot it against the number of degrees of freedom on log axes and read 
off the slope: that is the observed order of convergence.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.loglog(table['ndof'], table['eps_w [%]'], 'o-', label='central deflection')
ax.loglog(table['ndof'], table['eps_sigma [%]'], 's-', label='edge stress')
ax.set_xlabel('degrees of freedom'); ax.set_ylabel('relative error [%]')
ax.legend(frameon=False); ax.grid(alpha=0.3, which='both'); plt.show()


## The error floor, and where it comes from

Keep refining and the error stops falling. It does not go to zero, and the reason is not the 
mesh.

The analytical reference uses tabulated coefficients, $\alpha = 0.00126$ and $\beta = 0.0513$, 
quoted to three significant figures. Three figures is a relative precision of a few parts in 
a thousand. Once your discretisation error drops below that, you are no longer measuring the 
mesh — you are measuring the precision of the number you are comparing against.

Richardson extrapolation separates the two: extrapolate your own sequence to its limit, and 
compare that limit with the tabulated value.


In [ ]:
from memslab.analysis import richardson_limit

w_series = list(table['w_max_um'])
w_limit = richardson_limit(w_series)
w_ref = d.w_max(q) * 1e6

print(f'computed sequence      : {[round(v, 4) for v in w_series]}')
print(f'Richardson limit       : {w_limit:.4f} um')
print(f'tabulated reference    : {w_ref:.4f} um')
print(f'implied alpha          : {w_limit*1e-6*d.D/(q*d.a**4):.6f}')
print(f'tabulated alpha        : 0.00126')

# TASK: how many significant figures of agreement can this comparison support?
# TASK: what would you need in order to claim better agreement than that?


## Reflection

1. Your simulation and the textbook disagree at the fourth significant figure. Which one is 
   wrong? Defend your answer.
2. You are asked to certify this diaphragm to a 1% tolerance on sensitivity. Which mesh do 
   you choose, and what evidence do you attach?
3. Verification and validation are different things. Which of the two did the convergence 
   study do, and which did the comparison with theory do?
